# Exercises

In this section we have two exercises:
1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [1]:
# added 'd' for convenience
def build_kernel(data_set, kernel_type='linear', d=3):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        kernel = kernel ** d
    return kernel

In [2]:
import numpy as np
from sklearn.metrics.pairwise import polynomial_kernel

np.random.seed(42) 
X = np.random.rand(150, 10) * 5 

K_custom = build_kernel(X, kernel_type='poly', d=3)
K_sklearn = polynomial_kernel(X, degree=3, gamma=1.0, coef0=0.0)

print(f"Dataset shape: {X.shape}")
print(f"Kernel shape:  {K_custom.shape}\n")

print("My kernel (Top-left 3x3 slice):")
print(K_custom[:3, :3])

print("\nScikit kernel (Top-left 3x3 slice):")
print(K_sklearn[:3, :3])

if np.allclose(K_custom, K_sklearn):
    print("\n good")
else:
    print("\n bad")

Dataset shape: (150, 10)
Kernel shape:  (150, 150)

My kernel (Top-left 3x3 slice):
[[731044.36279364 294517.52850956  96483.03177173]
 [294517.52850956 212033.54896591  33998.27059759]
 [ 96483.03177173  33998.27059759 143736.28351031]]

Scikit kernel (Top-left 3x3 slice):
[[731044.36279364 294517.52850956  96483.03177173]
 [294517.52850956 212033.54896591  33998.27059759]
 [ 96483.03177173  33998.27059759 143736.28351031]]

 good


## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:
- ``choose_set_for_label``
- ``get_labels_count``

In [3]:
import cvxopt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
def choose_set_for_label(data_set, label):
    X, y = data_set
    y_bin = np.where(y == label, 1.0, -1.0)
    
    train_data_set, test_data_set, train_labels, test_labels = train_test_split(
        X, y_bin, test_size=0.2, random_state=42
    )

    train_labels = train_labels.reshape(-1, 1)
    test_labels = test_labels.reshape(-1, 1)
    
    return train_data_set, test_data_set, train_labels, test_labels

In [5]:
def get_labels_count(data_set):
    X, y = data_set
    labels_count = len(np.unique(y))
    return labels_count

Use the code that we have implemented earlier:

In [6]:
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

# !!! I make a small change here - introducing poly kernel for a test
def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        kernel = kernel ** 3
    return kernel

def classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    #K = np.dot(test_data_set, support_vectors.T)
    #kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (1. / sigma * np.sum(test_data_set ** 2, axis=1) * np.ones((1, np.shape(test_data_set)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    #aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (np.diag(np.dot(train_data_set, train_data_set.T))*np.ones((1,len(train_data_set)))).T[support_vectors_id]
    aa = np.dot(sv,np.ones((1,np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

## I'm curious how other kernels manage

In [7]:
def classify_linear(test_data_set, lambdas, targets, b, vector_number, support_vectors):
    kernel = np.dot(test_data_set, support_vectors.T)

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
        
    return np.sign(y)

def classify_poly(test_data_set, lambdas, targets, b, vector_number, support_vectors):
    kernel = np.dot(test_data_set, support_vectors.T) ** 3

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
        
    return np.sign(y)

In [8]:
iris = datasets.load_iris()
data_set = (iris.data, iris.target)

labels_count = get_labels_count(data_set)

_, _, _, true_test_labels = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)
predictions = np.zeros((len(true_test_labels), labels_count))

for label in range(labels_count):
    train_data_set, test_data_set, train_labels, test_labels = choose_set_for_label(data_set, label)
    
    objects_count = len(train_data_set)

    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(
        train_data_set, train_labels, kernel_type='rbf'
    )

    predicted = classify_rbf(
        test_data_set, train_data_set, lambdas, targets, b, 
        vector_number, support_vectors, support_vectors_id
    )
    
    predictions[:, label] = predicted.flatten()

predicted_multiclass = np.argmax(predictions, axis=1)

print("Accuracy Score:", accuracy_score(predicted_multiclass, true_test_labels))

     pcost       dcost       gap    pres   dres
 0:  1.3543e+02 -1.9835e+03  4e+03  2e-01  3e-15
 1:  8.6439e+01 -2.2486e+02  4e+02  9e-03  3e-15
 2:  1.2484e+01 -2.8119e+01  4e+01  2e-05  2e-15
 3: -7.5365e-01 -6.2783e+00  6e+00  1e-15  1e-15
 4: -2.0516e+00 -3.8076e+00  2e+00  2e-16  6e-16
 5: -2.5242e+00 -3.0976e+00  6e-01  4e-16  4e-16
 6: -2.7057e+00 -2.9450e+00  2e-01  2e-16  5e-16
 7: -2.7722e+00 -2.8528e+00  8e-02  4e-16  5e-16
 8: -2.7947e+00 -2.8064e+00  1e-02  2e-16  6e-16
 9: -2.7988e+00 -2.7992e+00  4e-04  8e-16  5e-16
10: -2.7989e+00 -2.7989e+00  4e-06  3e-16  5e-16
11: -2.7989e+00 -2.7989e+00  4e-08  2e-16  5e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0:  1.8559e+02 -4.6699e+03  9e+03  3e-01  5e-15
 1:  1.2358e+02 -5.4904e+02  7e+02  5e-03  4e-15
 2: -3.7544e+01 -2.0766e+02  2e+02  8e-04  5e-15
 3: -7.7104e+01 -1.2468e+02  5e+01  2e-04  5e-15
 4: -8.6869e+01 -1.1035e+02  2e+01  5e-05  5e-15
 5: -9.1449e+01 -1.0378e+02  1e+01  2e-05  6e-1

### Now I just do a simple test to compare kernels accuracy

In [9]:
iris = datasets.load_iris()
data_set = (iris.data, iris.target)

labels_count = get_labels_count(data_set)

_, _, _, true_test_labels = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

predictions = np.zeros((len(true_test_labels), labels_count))

for label in range(labels_count):
    train_data_set, test_data_set, train_labels, test_labels = choose_set_for_label(data_set, label)
    
    objects_count = len(train_data_set)

    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(
        train_data_set, train_labels, kernel_type='poly'
    )

    predicted = classify_poly(
        test_data_set, lambdas, targets, b, 
        vector_number, support_vectors
    )
    
    predictions[:, label] = predicted.flatten()

predicted_multiclass = np.argmax(predictions, axis=1)

print("Accuracy Score:", accuracy_score(predicted_multiclass, true_test_labels))

     pcost       dcost       gap    pres   dres
 0: -3.2393e+01 -6.0834e+03  2e+04  7e-01  1e-09
 1: -7.1186e+00 -1.4443e+03  2e+03  7e-02  1e-09
 2:  1.9744e-01 -1.5272e+02  3e+02  7e-03  3e-10
 3:  8.2622e-02 -1.4317e+01  2e+01  5e-04  3e-11
 4:  5.4150e-02 -2.1050e+00  3e+00  7e-05  4e-12
 5:  1.2925e-02 -4.7247e-01  7e-01  1e-05  7e-13
 6:  4.3466e-03 -6.2855e-02  9e-02  1e-06  1e-13
 7:  1.3047e-03 -2.5607e-03  4e-03  1e-08  2e-14
 8:  7.6241e-05 -4.1528e-04  5e-04  2e-16  9e-15
 9: -7.3237e-05 -1.5889e-04  9e-05  2e-16  4e-15
10: -1.2978e-04 -1.7770e-04  5e-05  4e-16  7e-15
11: -1.4477e-04 -1.4931e-04  5e-06  2e-16  8e-15
12: -1.4669e-04 -1.4701e-04  3e-07  2e-16  9e-15
13: -1.4685e-04 -1.4686e-04  9e-09  2e-16  8e-15
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -1.2238e+02 -9.0465e+03  3e+04  9e-01  3e-09
 1: -7.4367e+01 -3.7253e+03  7e+03  2e-01  3e-09
 2: -4.1834e+01 -1.0485e+03  2e+03  5e-02  2e-09
 3: -2.7895e+01 -3.9751e+02  6e+02  1e-02  1e-0

In [10]:
iris = datasets.load_iris()
data_set = (iris.data, iris.target)

labels_count = get_labels_count(data_set)

_, _, _, true_test_labels = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

predictions = np.zeros((len(true_test_labels), labels_count))

for label in range(labels_count):
    train_data_set, test_data_set, train_labels, test_labels = choose_set_for_label(data_set, label)
    
    objects_count = len(train_data_set)

    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(
        train_data_set, train_labels, kernel_type='linear'
    )

    predicted = classify_linear(
        test_data_set, lambdas, targets, b, 
        vector_number, support_vectors
    )

    predictions[:, label] = predicted.flatten()

predicted_multiclass = np.argmax(predictions, axis=1)

print("Accuracy Score:", accuracy_score(predicted_multiclass, true_test_labels))

     pcost       dcost       gap    pres   dres
 0: -2.9794e+01 -6.1637e+03  2e+04  7e-01  3e-13
 1:  2.5786e+01 -1.2748e+03  2e+03  6e-02  3e-13
 2:  2.2258e+01 -1.4611e+02  2e+02  6e-03  7e-14
 3:  6.4839e+00 -1.1472e+01  2e+01  7e-05  2e-14
 4:  4.2060e-01 -2.0135e+00  2e+00  2e-16  6e-15
 5: -2.6046e-01 -9.2188e-01  7e-01  2e-16  4e-15
 6: -4.9055e-01 -8.3550e-01  3e-01  2e-16  7e-15
 7: -6.4963e-01 -8.1956e-01  2e-01  2e-16  1e-14
 8: -7.3049e-01 -7.6818e-01  4e-02  2e-16  2e-14
 9: -7.4779e-01 -7.4833e-01  5e-04  2e-16  2e-14
10: -7.4805e-01 -7.4806e-01  8e-06  3e-16  1e-14
11: -7.4806e-01 -7.4806e-01  8e-08  2e-16  1e-14
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -3.8694e+02 -9.5660e+03  2e+04  6e-01  8e-13
 1: -3.8237e+02 -2.4892e+03  2e+03  3e-15  6e-13
 2: -5.0939e+02 -9.2066e+02  4e+02  1e-14  6e-13
 3: -6.1651e+02 -8.0056e+02  2e+02  9e-15  7e-13
 4: -6.5403e+02 -7.4739e+02  9e+01  2e-16  7e-13
 5: -6.7838e+02 -7.0985e+02  3e+01  2e-15  8e-1